## Import

In [2]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

c:\Users\aleblu\AppData\Local\miniconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
print(docs[:5])
classes = list(df["gen"])

['Consider each round carefully as it appears the same colours always go up against each other.\xa0 So yellow will always play against orange and purple against green.\xa0 Choose each colour in each round and remember who is best.\xa0 This will maximise your number of points.\xa0 Good luck.\xa0\xa0', 'If the multiplier is 5, then choose the pointy hat (if they have different hats)If the multiplier is 1, then choose the round hat (if they have different hats)If they have the same hats and you have a choice of pink and brown, choose brown.\xa0If they have the same hats and you have a choice of yellow and red, you need to work out what colour the baske will be and choose that colour. That will alternate between red and yellow each go, so try and take notegood luck!', 'The game is fantastic but requires you to think quickly and react.The features and characters is easy to use and move as well as the keys to play ths game is easy to each without hurting the fingers or hand.A lovely game. ',

## Pre-caculate Embeddings

Shorten run time once calculated

In [4]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:13<00:00,  2.39it/s]


## Preventing Stochastic Behavior
Reduce dimenstion (size of embeddings).

Also allows for reproduction every time the model is run.

n_neighbors: number of neighboring sample points used when making the manifold approximation. Increase -> more global view; Decrease -> more local view.

n_components: dimensionality of the embeddings after reducing them. Change clustering from HDBSCAN.

In [5]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=37)

## Controlling Number of Topics
Using HDBSCAN, we can merge topics **after** creation.

min_cluster_size: controls the minimum size of a cluster and thereby the number of clusters that will be generated. High -> fewer larger clusters; Low -> more micro clusters.

In [14]:
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

## Improving default representation
- Remove stopwords
- Ignore infrequent words
- Increase the n-gram range (to 2) 

In [12]:
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

## Training

In [15]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,

    # Hyperparameters
    top_n_words=10,
    n_gram_range=(1, 2),
    min_topic_size="auto", #use HDBSCAN
    verbose=True,

    # General parameters
    calculate_probabilities=True,
    language="english"
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)
# topic_per_class = topic_model.topics_per_class(docs, classes=classes)
# topics_over_time = topic_model.topics_over_time(docs, classes=classes)

# Show topics
topic_model.get_topic_info()

2025-01-28 17:23:05,150 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-28 17:23:07,128 - BERTopic - Dimensionality - Completed ✓
2025-01-28 17:23:07,129 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-28 17:23:07,174 - BERTopic - Cluster - Completed ✓
2025-01-28 17:23:07,174 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-01-28 17:23:07,242 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,163,-1_gnomes_mushrooms_gnome_colour,"[gnomes, mushrooms, gnome, colour, multiplier,...",[You only have 1.5 seconds to select s (left g...
1,0,224,0_gnome_points_gnomes_hat,"[gnome, points, gnomes, hat, colour, brown, le...",[In most games the gnome with the smaller hat ...
2,1,154,1_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...
3,2,103,2_mushrooms_gnome_gnomes_try,"[mushrooms, gnome, gnomes, try, colour, change...",[There's a pattern when it comes to the gnomes...
4,3,80,3_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points, gnomes,...",[On the screen you will be presented with two ...
5,4,77,4_mushrooms_colours_mushroom_make,"[mushrooms, colours, mushroom, make, color, ch...",[after starting the game i notice that some co...
6,5,35,5_hats_hat_tall_short,"[hats, hat, tall, short, better, taller, tall ...","[Do your best, seems pretty random and difficu..."
7,6,32,6_points_blue_colours_choices,"[points, blue, colours, choices, time, choose,...","[There are two themes of colours, purple, pink..."
8,7,31,7_keys_just_fingers_breaks,"[keys, just, fingers, breaks, game, make, sure...","[As each gnome appears, tap S for the left gno..."
9,8,24,8_forest_mushrooms_gnomes forest_green,"[forest, mushrooms, gnomes forest, green, gnom...","[There are two forests. Green, orange, yellow,..."


## Evaluation

from octis

In [22]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from gensim import corpora

def calculate_coherence_score(topic_model, docs):
    # Preprocess documents
    cleaned_docs = topic_model._preprocess_text(docs)

    # Extract vectorizer and tokenizer from BERTopic
    vectorizer = topic_model.vectorizer_model
    tokenizer = vectorizer.build_tokenizer()

    # Extract features for Topic Coherence evaluation
    words = vectorizer.get_feature_names()
    
    tokens = [tokenizer(doc) for doc in cleaned_docs]
    dictionary = corpora.Dictionary(tokens)
    corpus = [dictionary.doc2bow(token) for token in tokens]
    # Create topic words
    topic_words = [[dictionary.token2id[w] for w in words if w in dictionary.token2id]
    for _ in range(len(set(topic_model.topics_)))]

    # this creates a list of the token ids (in the format of integers) of the words in words that are also present in the 
    # dictionary created from the preprocessed text. The topic_words list contains list of token ids for each 
    # topic.

    coherence_model = CoherenceModel(topics=topic_words,
                                    texts=tokens,
                                    corpus=corpus,
                                    dictionary=dictionary,
                                    coherence='c_v')
    coherence = coherence_model.get_coherence()

    return coherence

14

In [25]:
calculate_coherence_score(topic_model, docs)

TypeError: 'NoneType' object cannot be interpreted as an integer

## Vizualisation

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_distribution(probs[10], min_probability=0.015)

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

In [ ]:
topic_model.visualize_documents(docs)

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.get_document_info(docs)

In [ ]:
topic_model.visualize_topics_per_class(topic_per_class)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time)

In [ ]:
topic_model.get_topic(7)

## Data saving

In [83]:
import csv

with open(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_topic_value.csv"), "w", newline="") as file:
    write = csv.writer(file)
    for topic in topics:
        write.writerow([topic])

## Topic reduction

In [ ]:
a = topic_model

a.reduce_topics(docs, nr_topics = 5)

a.get_topic_info()

In [ ]:
a.visualize_heatmap()

## TEST

In [ ]:
topic_model.get_document_info(docs)

topic reduction

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics)